# 10b · Azure SQL → Bronze with Change Tracking

**Who this is for:** you, after notebook 03b (hospital activity and SQL changes).
**Runs on:** Databricks serverless (costs credits: keep it short, detach when done). Most cells read only Delta tables; the one cell that touches Azure SQL is switched off by default because it wakes the database (~1,800 of the 100,000 free vCore-seconds a month).

**By the end you will be able to:**
- explain how the ingestion job decides between a **full copy** and **only the changes** (`plan_reads`)
- read the **watermark** table and say where the next run will start
- see the T-SQL the job sends through the Unity Catalog connection with `remote_query`
- explain why the job is **at-least-once**, and what Silver must do about it

The production code is `databricks/src/heavden/ingestion/sql_source.py`; this notebook imports it rather than copying it.

## 1. Setup

In [ ]:
import sys

# Find the repo folder this notebook was deployed into, and import the real code from it.
notebook_path = (
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
repo_root = "/Workspace" + notebook_path.split("/notebooks/databricks/")[0]
sys.path.insert(0, f"{repo_root}/databricks/src")

from heavden.ingestion import sql_source
from heavden.ingestion.sql_source import Read

dbutils.widgets.text("catalog", "heavden_dev")
dbutils.widgets.text("sql_schema", "dev")
CATALOG = dbutils.widgets.get("catalog")
SQL_SCHEMA = dbutils.widgets.get("sql_schema")
print(repo_root, CATALOG, SQL_SCHEMA)

## 2. The idea: version numbers instead of timestamps

SQL Server **Change Tracking** gives every committed change a version number (database-wide, always increasing). For each table it remembers *which keys* changed at *which version*, for 7 days (`CHANGE_RETENTION`).

So the job never asks "what changed since 10:00?". It asks "what changed after version 10?". Two numbers matter:
- `CHANGE_TRACKING_CURRENT_VERSION()`: the latest version in the database
- the **minimum valid version** per table: changes older than this were cleaned up, so you can't read them any more

Our **watermark** is the version the last run read up to.

## 3. `plan_reads`: snapshot or changes?

`plan_reads` is pure Python: no Spark, no SQL. It turns the two numbers above, plus our watermarks, into one decision per table. That's why it's unit-tested offline (`tests/test_sql_source.py`).

Below we feed it made-up numbers: current version 100, changes kept from version 10.

In [ ]:
import datetime as dt

CREATED = dt.datetime(2026, 10, 9, 17, 41)  # when the SQL tables were (re)created
SAME = dict.fromkeys(sql_source.TABLES, CREATED)  # creation times recorded by the last run


def fake_versions(current=100, min_valid=10):
    return [
        {
            "current_version": current,
            "schema_name": SQL_SCHEMA,
            "table_name": t,
            "min_valid_version": min_valid,
            "table_created": CREATED,
        }
        for t in sql_source.TABLES
    ]


first_run = sql_source.plan_reads(SQL_SCHEMA, fake_versions(), watermarks={})
later_run = sql_source.plan_reads(
    SQL_SCHEMA, fake_versions(), dict.fromkeys(sql_source.TABLES, 40), SAME
)
stale = sql_source.plan_reads(
    SQL_SCHEMA, fake_versions(), dict.fromkeys(sql_source.TABLES, 40) | {"encounters": 5}, SAME
)

for name, reads in [
    ("first run", first_run),
    ("later run", later_run),
    ("stale encounters", stale),
]:
    print(name, [(r.table, r.mode, r.from_version, r.to_version) for r in reads[:3]], "...")
print([r.mode for r in stale if r.table == "encounters"])

- **First run:** no watermark → copy everything up to version 100.
- **Later run:** watermark 40 → read changes in (40, 100].
- **Stale watermark:** encounters last read at version 5, but changes before 10 were cleaned up → the changes in (5, 10] are gone, so only a full copy is safe.

### The third reason for a full copy: a recreated table

`generator.load --replace` drops and recreates the SQL tables. That records **no deletes** and **doesn't move the version number**, so the job would happily read "changes since 10" from a brand-new table. That's fine when the data is identical, but a row that vanished in the reload would live on in Bronze and Silver.

So each run also records the table's creation time (`sys.tables.create_date`). If it differs from the one stored with the watermark, the table was recreated and `plan_reads` takes a snapshot. (We found this on 9 Oct: the reload after the landing upload read `changes (10, 18]` instead of a snapshot.)

In [ ]:
later = CREATED + dt.timedelta(hours=2)  # encounters was dropped and recreated
now = [
    dict(r, table_created=later) if r["table_name"] == "encounters" else r for r in fake_versions()
]
reads = sql_source.plan_reads(SQL_SCHEMA, now, dict.fromkeys(sql_source.TABLES, 100), SAME)
print({r.table: r.mode for r in reads})

# Watermarks written before creation times were recorded (None): a recreation can't be ruled
# out, so the job copies every table once.
unknown = dict.fromkeys(sql_source.TABLES, None)
print(
    {
        r.mode
        for r in sql_source.plan_reads(
            SQL_SCHEMA, fake_versions(), dict.fromkeys(sql_source.TABLES, 100), unknown
        )
    }
)

## 4. The T-SQL the job sends

The job can't run T-SQL directly from Spark. It wraps it in `remote_query`, which sends the query to Azure SQL through the Unity Catalog connection `heavden_sql`; the SQL login lives in the connection, not in the code.

In [ ]:
print(sql_source.versions_query(), end="\n\n")
print(sql_source.snapshot_query(SQL_SCHEMA, "encounters"), end="\n\n")
print(sql_source.changes_query(SQL_SCHEMA, Read("encounters", "changes", 10, 25)))

Three details in `changes_query`:
- `CHANGETABLE(CHANGES ..., 10)` returns one row per changed key with its **latest** version and operation (`I`nsert, `U`pdate, `D`elete).
- It's **left-joined** to the table to get the row as it is *now*. A deleted row has nothing to join, so its columns come back NULL.
- `WHERE SYS_CHANGE_VERSION <= 25` stops at the version we recorded, so a row changed *during* the run is left for the next run instead of being half-counted.

The queries contain no quotes: they travel inside a Spark SQL string literal, and `_literal` refuses anything that could break out of it.

## 5. The watermark table

In [ ]:
display(
    spark.sql(f"""
    SELECT table_name, mode, from_version, to_version, rows, inserts, updates, deletes,
           ingest_ts, table_created
    FROM {CATALOG}.bronze.sql_watermarks
    WHERE source_schema = '{SQL_SCHEMA}'
    ORDER BY ingest_ts, table_name
""")
)

One row per table per run. After the first two runs you should see a `snapshot` up to version 10 for every table, then `changes` from 10 to 10 with 0 rows: nothing changed in SQL, so nothing was copied.

The **latest row per table is the watermark**. The job reads it with `QUALIFY row_number() ...`:

In [ ]:
watermarks, created = sql_source.read_watermarks(spark, CATALOG, SQL_SCHEMA)
print(watermarks)
print(created)  # each table's creation time, as recorded by the last run

## 6. What landed in Bronze

In [ ]:
rows = []
for table in sql_source.TABLES:
    df = spark.table(f"{CATALOG}.bronze.sql_{table}")
    for r in df.groupBy("_change_op").count().collect():
        rows.append((table, r["_change_op"], r["count"]))
display(spark.createDataFrame(rows, "table string, change_op string, rows long"))

In [ ]:
display(spark.table(f"{CATALOG}.bronze.sql_encounters").limit(5))

Each Bronze row is the source row plus three columns:
- `_change_version`: the version of the change (for a snapshot: the version the copy was taken at)
- `_change_op`: `S` snapshot, `I` insert, `U` update, `D` delete
- `_ingest_ts`: when this run wrote it (the same value for every row of one run)

Bronze is **append-only**: an updated encounter (e.g. a discharge) appears again as a new row; nothing is overwritten. Silver turns this history into SCD2.

## 7. Optional: ask Azure SQL directly (wakes the database)

Leave `TOUCH_SOURCE = False` unless the database is already awake (the generator or the job ran in the last hour), or you're happy to spend one of the ~50 monthly wake-ups. The first query may take a minute while the database resumes; `read_versions` retries for you.

In [ ]:
TOUCH_SOURCE = False

if TOUCH_SOURCE:
    versions = sql_source.read_versions(spark, "heavden_sql", "heavden")
    current = versions[0]["current_version"]
    print("current version:", current)
    for r in versions:
        if r["schema_name"] == SQL_SCHEMA:
            print(f"  {r['table_name']:20} changes kept from version {r['min_valid_version']}")
    reads = sql_source.plan_reads(
        SQL_SCHEMA, versions, *sql_source.read_watermarks(spark, CATALOG, SQL_SCHEMA)
    )
    print("the next run would do:", {(r.mode, r.from_version, r.to_version) for r in reads})

## 8. At-least-once, and why that's fine

The job writes the Bronze rows **first**, and then the watermark. If it dies between the two, the next run starts from the old watermark and copies those changes again. Changes are never lost, but some may arrive twice: **at-least-once**.

Exactly-once would need both writes in a single transaction, and Delta can't do that across two tables. The standard answer is to make the next layer tolerant: Silver de-duplicates on (key, `last_updated`). The cell below is the check Silver will rely on.

In [ ]:
display(
    spark.sql(f"""
    SELECT encounter_id, last_updated, count(*) AS copies
    FROM {CATALOG}.bronze.sql_encounters
    GROUP BY encounter_id, last_updated
    HAVING count(*) > 1
""")
)  # copies > 1: the row landed more than once (a re-snapshot after a reload, or a run
# that died between its two writes). Expected: Silver keeps one version per change.

## 9. Try it yourself

**a)** Without running it: the current version is 60, `encounters` keeps changes from version 30, its watermark is 25, and it hasn't been recreated. What does `plan_reads` choose for `encounters`, and why? Then check with `fake_versions`.

In [ ]:
# Try it yourself
choice = None

In [ ]:
# Answer (run after trying)
reads = sql_source.plan_reads(
    SQL_SCHEMA, fake_versions(current=60, min_valid=30), dict.fromkeys(sql_source.TABLES, 25), SAME
)
choice = next(r for r in reads if r.table == "encounters")
# snapshot: changes in (25, 30] were cleaned up, so reading from 25 would silently miss them
print(choice)

**b)** Write a query that returns, for each encounter, only its **latest** Bronze row (the newest `_change_version`, then the newest `_ingest_ts`). That's the core of what Silver will do.

In [ ]:
# Try it yourself
latest_encounters_sql = None

In [ ]:
# Answer (run after trying)
latest_encounters_sql = f"""
    SELECT * FROM {CATALOG}.bronze.sql_encounters
    QUALIFY row_number() OVER (
        PARTITION BY encounter_id ORDER BY _change_version DESC, _ingest_ts DESC
    ) = 1
"""
print(spark.sql(latest_encounters_sql).count(), "encounters")

**c)** The generator deletes nothing: a discharge is an *update*. What would go wrong in Silver if a source row were deleted and Bronze only saw `I`/`U` rows? (Hint: look at how a deleted row comes back from the left join in section 4, and what the job does when `deletes > 0`.)

## 10. Pitfalls

**Pitfall: lazy doesn't mean "no I/O".** `spark.sql("SELECT * FROM remote_query(...)")` connects to Azure SQL *immediately*, to learn the columns. Our first real run failed because the retry wrapped `.collect()` but not `spark.sql()`. Anything that can fail on connect must sit inside the retry.

**Pitfall: a watermark older than the retention.** Change Tracking keeps 7 days. A dev catalog left untouched for longer will take a full snapshot on its next run. That's correct, but expect more duplicate rows in Bronze.

**Pitfall: raw JDBC on serverless.** `spark.read.format("jdbc")` isn't a supported source on serverless compute. Use a Unity Catalog connection (`remote_query`, or a foreign catalog).

**Next:** notebook 11 (Auto Loader) brings in the other kind of source: vitals and outcomes as JSON **files** in the `landing` volume.